In [49]:
from langchain.llms import OpenAI

temperature value--> how creative we want our model to be

0 --> temperature it means model is very safe it is not taking any bets 
1 --> it will take risk might generate wrong output but is very creative

In [ ]:
import os
os.environ["OPEN_API_KEY"]=""

In [51]:
llm=OpenAI(openai_api_key=os.environ["OPEN_API_KEY"],temperature=0.6)

In [18]:
text="What is the capital of India"
print(llm.predict(text))

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

In [ ]:
import os
os.environ["HUGGINGFACEHUB_API_TOKEN"] = ""

In [20]:
from langchain import HuggingFaceHub
llm_huggingface=HuggingFaceHub(repo_id="google/flan-t5-large", model_kwargs={"temperature":0, "max_length":64})

In [22]:
output=llm_huggingface.predict("Can you write a poem about AI")
print(output)

i love the way i look at the world i love the way i feel i love the way i think i feel i love the way i feel i love the way i think i feel i love the way i feel i love the way 


### Prompt Templates and LLM Chain

In [23]:
from langchain.prompts import PromptTemplate

prompt_template = PromptTemplate(input_variables=['country'], template="Tell me the capital of {country}")

prompt_template.format(country="India")

'Tell me the capital of India'

In [24]:
from langchain.chains import LLMChain
chain=LLMChain(llm=llm_huggingface, prompt=prompt_template)
print(chain.run("Australia"))

adelaide


### Combine Multiple Chains Using simple Sequential Chains

In [25]:
capital_template=PromptTemplate(input_variables=['country'], template="Please tell me the capital of {country}")
capital_chain=LLMChain(llm=llm_huggingface, prompt=capital_template)

famous_template=PromptTemplate(input_variables=['capital'], template="Recommend me places to visit in {capital}")

In [26]:
famous_chain=LLMChain(llm=llm_huggingface, prompt=famous_template)

In [27]:
from langchain.chains import SimpleSequentialChain
chain=SimpleSequentialChain(chains=[capital_chain, famous_chain])
chain.run("India")

'The National Museum of India'

### Sequential Chain

In [28]:
capital_template=PromptTemplate(input_variables=['country'], template="Please tell me the capital of {country}")
capital_chain=LLMChain(llm=llm_huggingface, prompt=capital_template, output_key="capital")

In [29]:
famous_template=PromptTemplate(input_variables=['capital'], template="Recommend me places to visit in {capital}")

famous_chain=LLMChain(llm=llm_huggingface, prompt=famous_template, output_key="places")

In [31]:
from langchain.chains import SequentialChain
chain=SequentialChain(chains=[capital_chain, famous_chain], input_variables=['country'], output_variables=['capital','places'])

In [32]:
chain({'country': "India"})

/var/folders/7q/rmzt6ycx62j1m9j573c3f60c0000gn/T/ipykernel_53619/79562688.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chain({'country': "India"})


{'country': 'India',
 'capital': 'chennai',
 'places': 'The National Museum of India'}

### Chatmodels with ChatOpenAI

In [33]:
from langchain.chat_models import ChatOpenAI

In [34]:
from langchain.schema import HumanMessage, SystemMessage, AIMessage

In [36]:
chatllm=ChatOpenAI(openai_api_key=os.environ["OPEN_API_KEY"],temperature=0.6, model='gpt-3.5-turbo')

SystemMessage - instructing the AI to behave in some way\
HumanMessage - input\
AIMessage - output

In [38]:
chatllm([
    SystemMessage(content="You are a comedian AI Assistant"),
    HumanMessage(content="Please provide some comedy punchlines on AI")
])

/var/folders/7q/rmzt6ycx62j1m9j573c3f60c0000gn/T/ipykernel_53619/637877644.py:1: LangChainDeprecationWarning: The method `BaseChatModel.__call__` was deprecated in langchain-core 0.1.7 and will be removed in 1.0. Use :meth:`~invoke` instead.
  chatllm([


RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### Prompt Template + LLM + OutputParser

In [39]:
from langchain.chat_models import ChatOpenAI
from langchain.prompts.chat import ChatPromptTemplate
from langchain.schema import BaseOutputParser

In [40]:
class Commaseperatedoutput(BaseOutputParser):
    def parse(self,text:str):
        return text.strip().split(",")

In [42]:
template = "You are a helpful assistant. When the user gives any input, you should generate five synonyms in a comma seperated list."
human_template = "{text}"
chatprompt = ChatPromptTemplate.from_messages([
    ("system", template),
    ("human", human_template)

])

In [43]:
chain=chatprompt|chatllm|Commaseperatedoutput()

In [45]:
chain.invoke({"text":"intelligent"})

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}